# Explore whisper models

In [ ]:
import pathlib
import whisper
import audio_transcription
whisper.available_models()

In [ ]:
mic_audio_filename = "microphone_audio.wav"
desktop_audio_filename = "desktop_audio.wav"
output_dir = f"./2024-07-12-output"
whisper_models = ['tiny','tiny.en','base','base.en','medium','large']

#setup folder
output_path = pathlib.Path(output_dir)
if not output_path.exists():
    output_path.mkdir()
print(f"Output dir is: {output_path}")

# -- Transcribe audio
output_path = pathlib.Path(output_dir)
mic_audio_path = output_path / mic_audio_filename
desktop_audio_path = output_path / desktop_audio_filename

for whisper_model in whisper_models:
    print(f"Transcribing with: {whisper_model}")
    try:
        audio_transcription.process_audio_files(
            str(mic_audio_path), 
            str(desktop_audio_path), 
            output_dir=str(output_path),
            whisper_model_name=whisper_model
        )
    except Exception as e:
        logger.error(f"An error occurred during the transcription process: {str(e)}")
        logger.error("\n\n Continuing to next model")

# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pathlib
import os
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

#local functions
import video_extraction
logger = video_extraction.logger

import audio_transcription

import transcript_merger

import summarize_transcript


from IPython.display import Audio, display
import ipywidgets as widgets
import pathlib





def get_audio_files_from_dirpath(path, suffixes=None):
    if suffixes is None:
        suffixes = ['.wav']
    # Usage example
    file_paths = [
        str(p) for p in pathlib.Path(path).iterdir()
        if p.suffix in suffixes
    ]  
    return file_paths

def get_audio_files_from_dirpath(path, suffixes=None):
    if suffixes is None:
        suffixes = ['.wav']
    # Usage example
    file_paths = [
        str(p) for p in pathlib.Path(path).iterdir()
        if p.suffix in suffixes
    ]  
    return file_paths
    
def play_audio_from_path(file_path):
    """
    Create an audio player widget for the given audio file path.
    
    :param file_path: Path to the audio file
    :return: IPython Audio widget
    """
    audio = Audio(filename=file_path)
    return audio

def create_audio_players(file_paths):
    """
    Create audio players for multiple file paths.
    
    :param file_paths: List of paths to audio files
    :return: List of (file name, audio player) tuples
    """
    players = []
    for path in file_paths:
        file_name = path.split('/')[-1]  # Extract file name from path
        player = play_audio_from_path(path)
        players.append((file_name, player))
    return players


In [ ]:
input_mkv = "./data/2024-07-10_16-48-55.mkv"
output_dir = "./output_test"

# run script
output_path = pathlib.Path(output_dir)
if not output_path.exists():
    output_path.mkdir()
    
result = video_extraction.process_mkv_file(input_mkv, output_dir)

# Usage example
file_paths = get_audio_files_from_dirpath(output_path)
audio_players = create_audio_players(file_paths)
for file_name, player in audio_players:
    print(f"Audio: {file_name}")
    display(player)

In [ ]:
mic_audio_filename = "microphone_audio.wav"
desktop_audio_filename = "desktop_audio.wav"

# script
mic_audio_path = output_path / mic_audio_filename
desktop_audio_path = output_path / desktop_audio_filename

try:
    audio_transcription.process_audio_files(str(mic_audio_path), str(desktop_audio_path), output_dir=str(output_path))
except Exception as e:
    logger.error(f"An error occurred during the transcription process: {str(e)}")

In [ ]:
mic_transcript_filename = "microphone_transcription.txt"
desktop_transcript_filename = "desktop_transcription.txt"
transcript_filename = "merged_transcript.json"
# script
mic_transcript_path = output_path / mic_transcript_filename
desktop_transcript_path = output_path / desktop_transcript_filename
transcript_file_path = output_path / transcript_filename

# Example usage

transcript_json = transcript_merger.process_transcripts(
    mic_transcript_file=str(mic_transcript_path), 
    desktop_transcript_file=str(desktop_transcript_path), 
    output_file=str(transcript_file_path),
    parse_overlaps=False
)

In [ ]:
summary_filename="haiku-summary.txt"
summarize=True #True to call the llm

# Main workflow
summary_filepath = output_path / summary_filename

transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model='claude-3-5-sonnet'
)

if summarize:
    try:
        summary = summarize_transcript.generate_summary(transcript_text, model_name='claude-3-haiku')
        summarize_transcript.save_summary(
            summary, 
            output_file=str(summary_filepath)
        )
        logger.info(f"Summary generated and saved to {summary_filepath}")
    except Exception as e:
        logger.error(f"An error occurred during summarization: {str(e)}")
    logger.info(summary)
else:
    logger.info("Skipping summarize step")

# Extract Audio Files

In [ ]:
#extract audio files
input_mkv = "./data/test.mkv"
output_dir = "./output"
result = video_extraction.process_mkv_file(input_mkv, output_dir)

# Transcribe audio

In [ ]:
import audio_transcription

# Usage
mic_audio_path = "./output/microphone_audio.wav"
desktop_audio_path = "./output/desktop_audio.wav"
output_dir = "./output"

try:
    audio_transcription.process_audio_files(mic_audio_path, desktop_audio_path, output_dir)
except Exception as e:
    logger.error(f"An error occurred during the transcription process: {str(e)}")

# Merge Transcripts

In [ ]:
import transcript_merger
# Example usage
mic_transcript_file = './output/microphone_transcription.txt'
desktop_transcript_file = './output/desktop_transcription.txt'
output_file = './output/merged_transcript.json'

transcript_json = transcript_merger.process_transcripts(
    mic_transcript_file, 
    desktop_transcript_file, 
    output_file,
    parse_overlaps=False
)

# Summarize Transcript

In [ ]:
import summarize_transcript
load_dotenv()

logger = summarize_transcript.logger
transcript_file="./output/merged_transcript.json"
output_file="./output/summary.txt"
summarize=False #True to call the llm

# Main workflow
transcript_data = summarize_transcript.load_transcript(transcript_file)
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(transcript_text, model='claude-3-5-sonnet')

if summarize:
    try:
        summary = summarize_transcript.generate_summary(transcript_text, model_name='claude-3-5-sonnet')
        summarize_transcript.save_summary(summary, output_file)
        logger.info(f"Summary generated and saved to {output_file}")
    except Exception as e:
        logger.error(f"An error occurred during summarization: {str(e)}")
else:
    logger.info("Skipping summarize step")